In [1]:
%pip install transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.6.0+cu124
12.4
True
1
NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import torch

print("Torch Version:", torch.__version__)
print("Torch CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch Version: 2.6.0+cu124
Torch CUDA Version: 12.4
CUDA Available: True
CUDA Device Count: 1
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer,AutoModel
sys.path.append(os.path.abspath(".."))
from shared import config
import torch
from demographic_module.trust_embedding import TrustEmbedding
from demographic_module.fusion_layer import FusionLayer


C:\Users\tngur\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
hate_df = pd.read_csv(config.PROCESSED_HATE_SPEECH_PATH)
hatexplain_df = pd.read_csv(config.PROCESSED_HATEXPLAIN_PATH)

print("Processed Hate Speech Dataset :", hate_df.shape)
print("Processed HateXplain Dataset :", hatexplain_df.shape)

Processed Hate Speech Dataset : (135556, 147)
Processed HateXplain Dataset : (20148, 7)


In [6]:
print("Hate Speech Dataset")
hate_df.head()

Hate Speech Dataset


,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,annotator_religion_nothing,annotator_religion_other,annotator_sexuality_bisexual,annotator_sexuality_gay,annotator_sexuality_straight,annotator_sexuality_other,label,clean_text,text_length,word_count
0,47777,10873,3,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,yes indeed she sort of reminds me of the elder...,239,51
1,39773,2790,2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,the trans women reading this tweet right now i...,57,10
2,47101,3379,3,4.0,4.0,4.0,4.0,4.0,4.0,0.0,...,True,False,False,False,True,False,hate,question these 4 broads who criticize america ...,231,43
3,43625,7365,3,2.0,3.0,2.0,1.0,2.0,0.0,0.0,...,False,False,False,False,True,False,normal,it is about time for all illegals to go back t...,123,24
4,12538,488,0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,False,False,False,False,True,False,hate,for starters bend over the one in pink and kic...,122,26


In [7]:
print("HateXplain Dataset")
hatexplain_df.head()

HateXplain Dataset


,post_id,text,labels,label,clean_text,text_length,word_count
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,"('normal', 'normal', 'normal')",normal,i dont think im getting my baby them white 9 h...,90,20
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,"('normal', 'normal', 'normal')",normal,we cannot continue calling ourselves feminists...,247,42
2,1178793830532956161_twitter,nawt yall niggers ignoring me,"('normal', 'normal', 'hatespeech')",normal,nawt yall niggers ignoring me,29,5
3,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,"('hatespeech', 'offensive', 'hatespeech')",hate,user i am bit confused coz chinese ppl can not...,116,23
4,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,"('hatespeech', 'hatespeech', 'offensive')",hate,this bitch in whataburger eating a burger with...,101,20


In [8]:
common_columns = list(set(hate_df.columns).intersection(hatexplain_df.columns))
print(common_columns)

['clean_text', 'text', 'word_count', 'text_length', 'label']


In [9]:
combined_df = pd.concat([hate_df[common_columns],hatexplain_df[common_columns]],ignore_index=True)

print("Combined Dataset Shape :", combined_df.shape)

Combined Dataset Shape : (155704, 5)


In [10]:
combined_df.info()
print(combined_df.head())

<class 'pandas.DataFrame'>
RangeIndex: 155704 entries, 0 to 155703
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   clean_text   155696 non-null  str  
 1   text         155704 non-null  str  
 2   word_count   155704 non-null  int64
 3   text_length  155704 non-null  int64
 4   label        155704 non-null  str  
dtypes: int64(2), str(3)
memory usage: 49.6 MB
                                          clean_text  \
0  yes indeed she sort of reminds me of the elder...   
1  the trans women reading this tweet right now i...   
2  question these 4 broads who criticize america ...   
3  it is about time for all illegals to go back t...   
4  for starters bend over the one in pink and kic...   

                                                text  word_count  text_length  \
0  Yes indeed. She sort of reminds me of the elde...          51          239   
1  The trans women reading this tweet right now i...          10    

In [11]:
print(combined_df["label"].value_counts())

label
normal       88438
hate         60867
offensive     6399
Name: count, dtype: int64


In [12]:
train_df, temp_df = train_test_split(combined_df,test_size=0.20,random_state=config.RANDOM_SEED,stratify=combined_df["label"])
valid_df, test_df = train_test_split(temp_df,test_size=0.50,random_state=config.RANDOM_SEED,stratify=temp_df["label"])

In [13]:
print("Training Samples :", len(train_df))
print("Validation Samples :", len(valid_df))
print("Testing Samples :", len(test_df))

Training Samples : 124563
Validation Samples : 15570
Testing Samples : 15571


In [14]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print("Tokenizer Loaded Successfully")

Tokenizer Loaded Successfully


In [15]:
print(dir(config))

['ANNOTATOR_WEIGHTS_PATH', 'BASE_DIR', 'BATCH_SIZE', 'DATA_DIR', 'HATEXPLAIN_DATASET_PATH', 'HATE_SPEECH_DATASET_PATH', 'LABEL_COLUMN', 'LEARNING_RATE', 'MAX_LENGTH', 'MODEL_NAME', 'NUM_EPOCHS', 'PROCESSED_DATA_DIR', 'PROCESSED_HATEXPLAIN_PATH', 'PROCESSED_HATE_SPEECH_PATH', 'Path', 'RANDOM_SEED', 'RAW_DATA_DIR', 'SOFT_LABELS_PATH', 'TEXT_COLUMN', 'TRAINING_DATASET_PATH', 'TRUST_MODULE_DIR', 'TRUST_PLOTS_DIR', 'TRUST_RESULTS_DIR', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']


In [16]:
label_mapping = {label: idx for idx, label in enumerate(sorted(combined_df["label"].unique()))}

print(label_mapping)

{'hate': 0, 'normal': 1, 'offensive': 2}


In [17]:
train_df["label_id"] = train_df["label"].map(label_mapping)
valid_df["label_id"] = valid_df["label"].map(label_mapping)
test_df["label_id"] = test_df["label"].map(label_mapping)

In [18]:
class HateSpeechDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["clean_text"].tolist()
        self.labels = dataframe["label_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = tokenizer(self.texts[index],truncation=True,padding="max_length",max_length=config.MAX_LENGTH,return_tensors="pt")
        
        return {"input_ids": encoding["input_ids"].squeeze(0),"attention_mask": encoding["attention_mask"].squeeze(0),"label":torch.tensor(self.labels[index],dtype=torch.long)}

In [19]:
train_dataset = HateSpeechDataset(train_df)
valid_dataset = HateSpeechDataset(valid_df)
test_dataset = HateSpeechDataset(test_df)

In [20]:
train_loader = DataLoader(train_dataset,batch_size=config.BATCH_SIZE,shuffle=True)

valid_loader = DataLoader(valid_dataset,batch_size=config.BATCH_SIZE)
test_loader = DataLoader(test_dataset,batch_size=config.BATCH_SIZE)

In [21]:
modernbert = AutoModel.from_pretrained(config.MODEL_NAME)
print("ModernBERT Loaded Successfully")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1942.40it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModernBERT Loaded Successfully


In [22]:
class ModernBERTClassifier(nn.Module):

    def __init__(self, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(config.MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size,num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        cls_embedding = self.dropout(cls_embedding)
        logits = self.classifier(cls_embedding)

        return logits

In [23]:
num_labels = len(label_mapping)
model = ModernBERTClassifier(num_labels)
print(model)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 2659.26it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModernBERTClassifier(
  (bert): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (attn_norm): La

In [24]:
batch = next(iter(train_loader))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
with torch.no_grad():
    logits = model(input_ids=input_ids,attention_mask=attention_mask)

print("Logits Shape :", logits.shape)

Logits Shape : torch.Size([16, 3])


In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=config.LEARNING_RATE)
print("Loss Function and Optimizer Created")

Loss Function and Optimizer Created


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using Device :", device)

Using Device : cuda


In [27]:
print("Combined Dataset :", len(combined_df))
print("Training Samples :", len(train_df))
print("Validation Samples :", len(valid_df))
print("Testing Samples :", len(test_df))

print("\nModernBERT Loaded")
print("Tokenizer Ready")
print("Dataset Ready")
print("DataLoader Ready")
print("Model ready for training")

Combined Dataset : 155704
Training Samples : 124563
Validation Samples : 15570
Testing Samples : 15571

ModernBERT Loaded
Tokenizer Ready
Dataset Ready
DataLoader Ready
Model ready for training


In [28]:
training_df = pd.read_csv(config.TRAINING_DATASET_PATH)

print("Training Dataset Loaded")
print(training_df.shape)

Training Dataset Loaded
(59713, 12)


In [29]:
training_df.head()

,item_id,dataset_source,text,clean_text,text_length,word_count,majority_label,predicted_label,confidence,p_normal,p_offensive,p_hate
0,47777,hate_speech,Yes indeed. She sort of reminds me of the elde...,yes indeed she sort of reminds me of the elder...,239,51,normal,normal,1.000000,1.000000e+00,NaN,6.422661e-13
1,39773,hate_speech,The trans women reading this tweet right now i...,the trans women reading this tweet right now i...,57,10,normal,normal,1.000000,1.000000e+00,NaN,8.661139e-31
2,47101,hate_speech,Question: These 4 broads who criticize America...,question these 4 broads who criticize america ...,231,43,hate,hate,1.000000,8.242883e-22,NaN,1.000000e+00
3,43625,hate_speech,It is about time for all illegals to go back t...,it is about time for all illegals to go back t...,123,24,normal,normal,1.000000,1.000000e+00,NaN,2.018769e-09
4,12538,hate_speech,For starters bend over the one in pink and kic...,for starters bend over the one in pink and kic...,122,26,hate,normal,0.841507,8.415066e-01,NaN,1.584934e-01


In [30]:
training_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59713 entries, 0 to 59712
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   item_id          59713 non-null  str    
 1   dataset_source   59713 non-null  str    
 2   text             59713 non-null  str    
 3   clean_text       59709 non-null  str    
 4   text_length      59713 non-null  int64  
 5   word_count       59713 non-null  int64  
 6   majority_label   59713 non-null  str    
 7   predicted_label  59713 non-null  str    
 8   confidence       59713 non-null  float64
 9   p_normal         59713 non-null  float64
 10  p_offensive      20148 non-null  float64
 11  p_hate           59713 non-null  float64
dtypes: float64(4), int64(2), str(6)
memory usage: 22.4 MB


In [31]:
print(training_df.columns)

Index(['item_id', 'dataset_source', 'text', 'clean_text', 'text_length',
       'word_count', 'majority_label', 'predicted_label', 'confidence',
       'p_normal', 'p_offensive', 'p_hate'],
      dtype='str')


In [32]:
print(training_df["predicted_label"].value_counts())

predicted_label
normal       37908
hate         18414
offensive     3391
Name: count, dtype: int64


In [33]:
print(training_df[["confidence","p_normal","p_offensive","p_hate"]].head())

   confidence      p_normal  p_offensive        p_hate
0    1.000000  1.000000e+00          NaN  6.422661e-13
1    1.000000  1.000000e+00          NaN  8.661139e-31
2    1.000000  8.242883e-22          NaN  1.000000e+00
3    1.000000  1.000000e+00          NaN  2.018769e-09
4    0.841507  8.415066e-01          NaN  1.584934e-01


In [34]:
train_ds_df, temp_ds_df = train_test_split(training_df,test_size=0.20,random_state=config.RANDOM_SEED,stratify=training_df["predicted_label"])

valid_ds_df, test_ds_df = train_test_split(temp_ds_df,test_size=0.50,random_state=config.RANDOM_SEED,stratify=temp_ds_df["predicted_label"])

In [35]:
print("Training :", len(train_ds_df))
print("Validation :", len(valid_ds_df))
print("Testing :", len(test_ds_df))

Training : 47770
Validation : 5971
Testing : 5972


In [36]:
class TrainingDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["clean_text"].tolist()
        self.labels = dataframe["label_id"].tolist()
        self.confidence = dataframe["confidence"].tolist()
        self.p_normal = dataframe["p_normal"].fillna(0).tolist()
        self.p_offensive = dataframe["p_offensive"].fillna(0).tolist()
        self.p_hate = dataframe["p_hate"].fillna(0).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = tokenizer(self.texts[index], truncation=True,           padding="max_length",max_length=config.MAX_LENGTH,          return_tensors="pt")

        return {
            "input_ids": encoding["input_ids"].squeeze(0),           "attention_mask": encoding["attention_mask"].squeeze(0),        "label": torch.tensor(self.labels[index], dtype=torch.long),    "confidence": torch.tensor(self.confidence[index], dtype=torch.float),  "p_normal": torch.tensor(self.p_normal[index], dtype=torch.float),  "p_offensive": torch.tensor(self.p_offensive[index], dtype=torch.float), "p_hate": torch.tensor(self.p_hate[index], dtype=torch.float)
        }



In [37]:
label_mapping = {
    label: idx
    for idx, label in enumerate(
        sorted(training_df["predicted_label"].unique())
    )
}

print(label_mapping)

{'hate': 0, 'normal': 1, 'offensive': 2}


In [38]:
train_ds_df["label_id"] = train_ds_df["predicted_label"].map(label_mapping)
valid_ds_df["label_id"] = valid_ds_df["predicted_label"].map(label_mapping)
test_ds_df["label_id"] = test_ds_df["predicted_label"].map(label_mapping)

In [39]:
print(train_ds_df[["predicted_label", "label_id"]].head())

      predicted_label  label_id
17736          normal         1
58640            hate         0
39630            hate         0
13894            hate         0
13726          normal         1


In [40]:
print(train_ds_df.columns)

Index(['item_id', 'dataset_source', 'text', 'clean_text', 'text_length',
       'word_count', 'majority_label', 'predicted_label', 'confidence',
       'p_normal', 'p_offensive', 'p_hate', 'label_id'],
      dtype='str')


In [41]:
print(TrainingDataset.__dict__.keys())

dict_keys(['__module__', '__init__', '__len__', '__getitem__', '__doc__', '__parameters__'])


In [42]:
train_ds_df["clean_text"] = train_ds_df["clean_text"].fillna("").astype(str)
valid_ds_df["clean_text"] = valid_ds_df["clean_text"].fillna("").astype(str)
test_ds_df["clean_text"] = test_ds_df["clean_text"].fillna("").astype(str)

In [43]:
train_dataset_v2 = TrainingDataset(train_ds_df)
valid_dataset_v2 = TrainingDataset(valid_ds_df)
test_dataset_v2 = TrainingDataset(test_ds_df)

In [44]:
train_loader_v2 = DataLoader(train_dataset_v2,batch_size=config.BATCH_SIZE,shuffle=True)

valid_loader_v2 = DataLoader(valid_dataset_v2, batch_size=config.BATCH_SIZE)
test_loader_v2 = DataLoader(test_dataset_v2, batch_size=config.BATCH_SIZE)

In [45]:
batch = next(iter(train_loader_v2))
print(batch.keys())

dict_keys(['input_ids', 'attention_mask', 'label', 'confidence', 'p_normal', 'p_offensive', 'p_hate'])


In [46]:
print("Input IDs :", batch["input_ids"].shape)
print("Attention Mask :", batch["attention_mask"].shape)
print("Labels :", batch["label"].shape)
print("Confidence :", batch["confidence"].shape)
print("P Normal :", batch["p_normal"].shape)
print("P Offensive :", batch["p_offensive"].shape)
print("P Hate :", batch["p_hate"].shape)

Input IDs : torch.Size([16, 128])
Attention Mask : torch.Size([16, 128])
Labels : torch.Size([16])
Confidence : torch.Size([16])
P Normal : torch.Size([16])
P Offensive : torch.Size([16])
P Hate : torch.Size([16])


In [47]:
print(batch["label"])
print(batch["confidence"])
print(batch["p_normal"])
print(batch["p_offensive"])
print(batch["p_hate"])

tensor([1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 0.9615, 0.9275, 0.9974, 1.0000, 1.0000, 0.9594])
tensor([1.0000e+00, 1.0000e+00, 2.3843e-21, 2.7012e-05, 3.8557e-10, 1.0000e+00,
        6.7177e-20, 1.0000e+00, 1.0000e+00, 8.0276e-18, 7.3221e-05, 8.1654e-03,
        9.9740e-01, 1.1078e-20, 1.0000e+00, 9.5936e-01])
tensor([0.0000e+00, 0.0000e+00, 0.0000e+00, 1.8418e-11, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 3.8462e-02, 6.4288e-02,
        7.3625e-06, 0.0000e+00, 0.0000e+00, 0.0000e+00])
tensor([7.5800e-11, 2.8236e-11, 1.0000e+00, 9.9997e-01, 1.0000e+00, 2.8204e-21,
        1.0000e+00, 4.9591e-14, 5.8087e-11, 1.0000e+00, 9.6146e-01, 9.2755e-01,
        2.5920e-03, 1.0000e+00, 1.2687e-12, 4.0643e-02])


In [48]:
class DemographicModernBERT(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(config.MODEL_NAME)
        self.trust_embedding = TrustEmbedding()
        self.fusion_layer = FusionLayer(bert_hidden_size=self.bert.config.hidden_size,demographic_size=64,output_size=self.bert.config.hidden_size)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size,num_labels)

    def forward(self,input_ids,attention_mask,confidence,p_normal,p_offensive,p_hate):
        outputs = self.bert(input_ids=input_ids,          attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]
        trust_vector = self.trust_embedding(confidence,p_normal,p_offensive,p_hate)

        fused_embedding = self.fusion_layer(cls_embedding,           trust_vector)

        fused_embedding = self.dropout(fused_embedding)
        logits = self.classifier(fused_embedding)

        return logits

In [49]:
num_labels = len(label_mapping)

model = DemographicModernBERT(
    num_labels=num_labels
)

print(model)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 2449.20it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DemographicModernBERT(
  (bert): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (attn_norm): L

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using Device :", device)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Using Device : cuda
GPU : NVIDIA GeForce RTX 4050 Laptop GPU


In [51]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=config.LEARNING_RATE)
print("Loss Function Ready")
print("Optimizer Ready")

Loss Function Ready
Optimizer Ready


In [52]:
batch = next(iter(train_loader_v2))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
confidence = batch["confidence"].to(device)
p_normal = batch["p_normal"].to(device)
p_offensive = batch["p_offensive"].to(device)
p_hate = batch["p_hate"].to(device)
labels = batch["label"].to(device)

In [53]:
with torch.no_grad():
    outputs = model(input_ids=input_ids,attention_mask=attention_mask,confidence=confidence,p_normal=p_normal,p_offensive=p_offensive,p_hate=p_hate)

print("Output Shape :", outputs.shape)

Output Shape : torch.Size([16, 3])


In [54]:
predictions = torch.argmax(outputs, dim=1)
print("Predictions")
print(predictions)

Predictions
tensor([1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0], device='cuda:0')


In [55]:
loss = criterion(outputs, labels)
print("Loss :", loss.item())

Loss : 1.0201053619384766


In [56]:
EPOCHS = 3
print("Number of Epochs :", EPOCHS)

Number of Epochs : 3


In [57]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        confidence = batch["confidence"].to(device)
        p_normal = batch["p_normal"].to(device)
        p_offensive = batch["p_offensive"].to(device)
        p_hate = batch["p_hate"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids,attention_mask=attention_mask,confidence=confidence,p_normal=p_normal,p_offensive=p_offensive,p_hate=p_hate)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / len(dataloader)
    epoch_accuracy = correct / total
    return epoch_loss, epoch_accuracy

In [58]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            confidence = batch["confidence"].to(device)
            p_normal = batch["p_normal"].to(device)
            p_offensive = batch["p_offensive"].to(device)
            p_hate = batch["p_hate"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids,attention_mask=attention_mask,confidence=confidence,p_normal=p_normal,p_offensive=p_offensive,p_hate=p_hate)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    validation_loss = total_loss / len(dataloader)
    validation_accuracy = correct / total
    return validation_loss, validation_accuracy

In [ ]:
train_losses = []
train_accuracies = []
validation_losses = []
validation_accuracies = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss, train_accuracy = train_one_epoch(model,train_loader_v2,optimizer,      criterion,device)

    validation_loss, validation_accuracy = validate(model,valid_loader_v2,criterion,       device)
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)
    print(f"Training Loss      : {train_loss:.4f}")
    print(f"Training Accuracy  : {train_accuracy:.4f}")
    print(f"Validation Loss    : {validation_loss:.4f}")
    print(f"Validation Accuracy: {validation_accuracy:.4f}")


Epoch 1/3
